# Bank Marketing Veri Tanıma

Bu not defteri hafta 2 için veri setinin ilk incelemesini yapar.

In [ ]:
df = load_raw()
print('Shape:', df.shape)
print(df.dtypes)
print(df.head())

In [ ]:
df.describe(include=[np.number]).T

In [ ]:
categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
for col in categorical_cols:
    print(f'\n{col}: {df[col].nunique()} unique values')
    print(df[col].value_counts().head(10).to_string())

In [ ]:
target_counts = df['y'].value_counts()
print(target_counts)
print(df['y'].value_counts(normalize=True)*100)

In [ ]:
unknown_counts = {col: int((df[col].astype(str) == 'unknown').sum()) for col in df.columns}
print(pd.Series(unknown_counts).sort_values(ascending=False))

In [ ]:
print('pdays == -1:', int((df['pdays'] == -1).sum()))
print('duplicate rows:', int(df.duplicated().sum()))
print(df.groupby('y')['duration'].mean())

In [ ]:
pdays_poutcome = pd.crosstab(df['pdays'] == -1, df['poutcome'], margins=True)
print(pdays_poutcome)
print('pdays == -1 & unknown:', int(((df['pdays'] == -1) & (df['poutcome'] == 'unknown')).sum()))
print('pdays != -1 & unknown:', int(((df['pdays'] != -1) & (df['poutcome'] == 'unknown')).sum()))
print('previous == 0 equals pdays == -1:', bool((df['previous'] == 0).equals(df['pdays'] == -1)))


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from src.config import DROP_COLS, RANDOM_STATE, TARGET, TEST_SIZE

X = df.drop(columns=DROP_COLS + [TARGET])
y = df[TARGET].map({'yes': 1, 'no': 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
model = DummyClassifier(strategy='most_frequent')
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]
metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred, zero_division=0),
    'recall': recall_score(y_test, y_pred, zero_division=0),
    'f1': f1_score(y_test, y_pred, zero_division=0),
    'roc_auc': roc_auc_score(y_test, y_proba),
    'pr_auc': average_precision_score(y_test, y_proba),
}
print(metrics)


In [ ]:
print(json.dumps(summary, ensure_ascii=False, indent=2))